# 第4回：データ探偵—分布・欠損・外れ値

**今日の問い：モデルを作る前に、データの怪しいところをどう見つけるか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 単変量・二変量・群別の順でデータを見る
- 欠損や外れ値を調査対象として扱う
- 図から断定ではなく検証可能な仮説を作る

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。

### 先に押さえる言葉

- 分布：値がどこにどれだけ存在するか
- 外れ値：他と大きく異なる観測値
- 相関：2変数が一緒に変化する程度
- EDA：モデル化前に品質と構造を探索する作業

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", font="sans-serif")
from matplotlib import font_manager
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for candidate in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP"]:
    if candidate in available_fonts:
        plt.rcParams["font.family"] = candidate
        break


## TRY：1変数の分布を見る


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="yield_pct", bins=20, ax=axes[0])
axes[0].set_title("収率の分布")
sns.boxplot(data=df, x="reaction_time_h", ax=axes[1])
axes[1].set_title("反応時間：外れ値候補を探す")
plt.tight_layout()


## 2変数の関係とカテゴリ比較


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="temperature_c", y="yield_pct", hue="catalyst", alpha=0.65, ax=axes[0])
axes[0].set_title("温度と収率")
sns.boxplot(data=df, x="catalyst", y="yield_pct", ax=axes[1])
axes[1].set_title("触媒別の収率")
plt.tight_layout()


## TRY：欠損と怪しい値を表で確認


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("欠損数"))
display(df.nlargest(5, "temperature_c")[["sample_id", "temperature_c", "reaction_time_h", "yield_pct"]])


## CHANGE

色分けを`catalyst`から`solvent`へ変えます。見え方が変わった点を1つ共有します。

## 注意

外れ値は入力ミスとは限りません。「誰に確認するか」「残す場合に何が起きるか」まで考えます。


## DEEP DIVE：結果を一段深く読む

次のセルは、数値を出して終わらず「どの条件で、なぜそう見えるか」を調べる発展です。

### 出力を見る観点

- 軸・単位・件数を確認してから形を見る
- 相関は非線形関係や群ごとの差を隠すことがある
- 欠損の発生理由が予測時点と関係するか考える


In [ ]:
numeric_columns = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa", "yield_pct"]
correlation = df[numeric_columns].corr()
plt.figure(figsize=(9, 6))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("数値列の相関（因果関係ではない）")
plt.tight_layout()
missing_by_solvent = df.assign(temperature_missing=df["temperature_c"].isna()).groupby("solvent", dropna=False)["temperature_missing"].agg(["count", "mean"])
display(missing_by_solvent.rename(columns={"count": "件数", "mean": "温度欠損率"}).round(3))


## よくある誤り

- 外れ値を自動削除する
- 相関を因果と読む
- 見栄えの良い図だけを選ぶ

## SELF-STUDY（任意・30〜60分）

- 数値列の相関ヒートマップから仮説を1つ書く
- 外れ値候補2件について確認先・残す場合・除く場合を整理する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 箱ひげ図で何が分かるか
2. 欠損率だけでは足りない理由は何か
3. 良い仮説に必要な次の確認は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
